In [1]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm

import torch._dynamo
torch._dynamo.config.suppress_errors = True



# Device & perf settings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")



# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   # update path
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Class names (first 10):", dataset.classes[:10], "...")

# ---- Stratified split (85/10/5) ----
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15,
                                           random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3,
                                         random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")



#  Model (DenseNet121)

weights = DenseNet121_Weights.DEFAULT
model = densenet121(weights=weights)

num_classes = len(dataset.classes)  # e.g., 200
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

model = model.to(device, memory_format=torch.channels_last)

if hasattr(torch, "compile"):
    model = torch.compile(model, mode="max-autotune")

print(model)



# Loss, Optimizer, Scheduler

criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4, nesterov=True)

epochs = 90
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)



# Training & Evaluation

def evaluate_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer,
                device, epochs=90, patience=10):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype,
                                enabled=(device.type=="cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100.0 * correct / total
        val_acc = evaluate_model(model, val_loader, device)
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {running_loss/total:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "densenet121_best.pth")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f" Early stopping at epoch {epoch+1}")
            break

        scheduler.step()



# 5. Run Training

epochs = 20
train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs)

# 6. Save Weights

torch.save(model.state_dict(), "densenet121_weights.pth")
print("Saved DenseNet121 weights")


# 7. Final Test Accuracy

test_acc = evaluate_model(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2f}%")


Using device: cuda
Total images: 100000
Number of classes: 200
Class names (first 10): ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
Train: 85000, Val: 10000, Test: 5000
✅ DataLoaders ready
OptimizedModule(
  (_orig_mod): DenseNet(
    (features): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, mome

AUTOTUNE convolution(128x3x224x224, 64x3x7x7)
  convolution 2.1064 ms 100.0%
  triton_convolution_3 2.8621 ms 73.6%
  triton_convolution_4 3.1304 ms 67.3%
  triton_convolution_5 3.4202 ms 61.6%
  triton_convolution_2 4.1441 ms 50.8%
  triton_convolution_0 5.2925 ms 39.8%
  triton_convolution_1 9.4141 ms 22.4%
SingleProcess AUTOTUNE takes 51.5550 seconds
AUTOTUNE mm(401408x64, 64x128)
  triton_mm_8 0.1280 ms 100.0%
  triton_mm_14 0.1280 ms 100.0%
  triton_mm_7 0.1290 ms 99.2%
  triton_mm_9 0.1290 ms 99.2%
  triton_mm_10 0.1290 ms 99.2%
  triton_mm_13 0.1300 ms 98.4%
  triton_mm_6 0.1331 ms 96.2%
  triton_mm_16 0.1475 ms 86.8%
  mm 0.1526 ms 83.9%
  triton_mm_15 0.1638 ms 78.1%
SingleProcess AUTOTUNE takes 4.2039 seconds
AUTOTUNE convolution(128x128x56x56, 32x128x3x3)
  convolution 0.2417 ms 100.0%
  triton_convolution_24 0.3369 ms 71.7%
  triton_convolution_21 0.3420 ms 70.7%
  triton_convolution_19 0.3932 ms 61.5%
  triton_convolution_22 0.4055 ms 59.6%
  triton_convolution_23 0.4342 m

Epoch [1/20] | Train Loss: 3.4085 | Train Acc: 33.28% | Val Acc: 27.45%
Epoch [2/20] | Train Loss: 2.6603 | Train Acc: 50.13% | Val Acc: 32.08%
Epoch [3/20] | Train Loss: 2.4234 | Train Acc: 56.34% | Val Acc: 45.52%
Epoch [4/20] | Train Loss: 2.2766 | Train Acc: 60.38% | Val Acc: 44.24%
Epoch [5/20] | Train Loss: 2.1724 | Train Acc: 63.56% | Val Acc: 49.34%
Epoch [6/20] | Train Loss: 2.0904 | Train Acc: 65.78% | Val Acc: 50.51%
Epoch [7/20] | Train Loss: 2.0231 | Train Acc: 67.64% | Val Acc: 48.67%
Epoch [8/20] | Train Loss: 1.9644 | Train Acc: 69.33% | Val Acc: 50.59%
Epoch [9/20] | Train Loss: 1.9118 | Train Acc: 70.95% | Val Acc: 49.12%
Epoch [10/20] | Train Loss: 1.8746 | Train Acc: 71.75% | Val Acc: 51.89%
Epoch [11/20] | Train Loss: 1.8346 | Train Acc: 73.34% | Val Acc: 54.79%
Epoch [12/20] | Train Loss: 1.7918 | Train Acc: 74.58% | Val Acc: 54.58%
Epoch [13/20] | Train Loss: 1.7606 | Train Acc: 75.29% | Val Acc: 54.18%
Epoch [14/20] | Train Loss: 1.7248 | Train Acc: 76.56% | Val